# Name That LLM: offline E24 external E0c replacement

This version changes only the E0c stream from the successful E16 submission. Its replacement was fitted on all 2,699 official training responses plus 3,620 fixed, provenance-audited Prvsiyan v9 responses covering 17 classes. The per-class external loss weight totals one quarter of the official class weight. Qwen has no verified external rows. RoBERTa, three ModernBERT seeds, row-wise z standardization, and equal stream weights are unchanged. The fixed 120-prompt fivefold OOF balanced accuracy was 0.425000, versus 0.409259 for E16. This exploratory result is not a Kaggle score: the external source protocol used global official training length statistics. Attach the existing E0c/RoBERTa, E16 ModernBERT, and offline wheelhouse inputs, plus the small E24 artifact. Source credit: [Prvsiyan Extended Name That LLM Dataset v9](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/version/9), CC BY 4.0. Internet off; accelerator None.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
COMPETITION_DIR = INPUT_ROOT / "competitions" / "name-that-llm"
ARTIFACT_ROOT = None  # Set to the mounted Kaggle Dataset directory if autodetection is ambiguous.
ARTIFACT_ZIP = None  # Optional path to an attached ZIP if multiple model ZIPs are mounted.
WHEELHOUSE_DIR = None  # Auto-filled from an attached extracted wheelhouse directory or ZIP.
WHEELHOUSE_ZIP = None  # Optional explicit ZIP path if autodetection is ambiguous.
OUTPUT_PATH = Path("/kaggle/working/submission.parquet")
MODERN_ARTIFACT_ROOT = None  # Optional mounted E16 Dataset directory override.
MODERN_ARTIFACT_ZIP = None  # Optional E16 ZIP override if autodetection is ambiguous.

E24_ARTIFACT_ROOT = None  # Optional mounted E24 Dataset directory override.
E24_ARTIFACT_ZIP = None  # Optional E24 ZIP override if autodetection is ambiguous.


In [ ]:
import hashlib
import json
import zipfile

RECIPE = "e0c_plus_roberta_12ep_scalerfix_equal_rank"
SOURCE_FILES = ("final_pipeline.py", "run_e0.py", "run_style.py",
                "run_combined.py", "run_fixed_blend.py")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

ARTIFACT_ARCHIVE = None
if ARTIFACT_ROOT is None:
    candidates = []
    for item in INPUT_ROOT.rglob("artifact_manifest.json"):
        try:
            candidate = json.loads(item.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if candidate.get("recipe") == RECIPE:
            candidates.append(item.parent)
    if not candidates:
        zip_candidates = []
        for item in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(item) as bundle:
                    if "artifact_manifest.json" in bundle.namelist():
                        zip_candidates.append(item)
            except (OSError, zipfile.BadZipFile):
                continue
        if ARTIFACT_ZIP is not None:
            zip_candidates = [Path(ARTIFACT_ZIP)]
        if len(zip_candidates) != 1:
            raise RuntimeError(
                f"Expected one model artifact ZIP, found {len(zip_candidates)}: {zip_candidates}. "
                "Set ARTIFACT_ZIP in the config cell."
            )
        ARTIFACT_ARCHIVE = Path(zip_candidates[0]).resolve(strict=True)
        extracted_root = Path("/kaggle/working/name_that_llm_artifact")
        with zipfile.ZipFile(zip_candidates[0]) as bundle:
            names = bundle.namelist()
            if len(names) != len(set(names)) or "artifact_manifest.json" not in names:
                raise RuntimeError("Invalid model artifact ZIP")
            if any(name.startswith("/") or ".." in name.split("/") or "\\" in name for name in names):
                raise RuntimeError("Unsafe path in model artifact ZIP")
            bundle.extractall(extracted_root)
        candidates = [extracted_root]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one model artifact Dataset, found {len(candidates)}: {candidates}. "
            "Set ARTIFACT_ROOT in the config cell."
        )
    ARTIFACT_ROOT = candidates[0]
ARTIFACT_ROOT = Path(ARTIFACT_ROOT).resolve(strict=True)
manifest = json.loads((ARTIFACT_ROOT / "artifact_manifest.json").read_text(encoding="utf-8"))
if manifest.get("format_version") != 1 or manifest.get("recipe") != RECIPE:
    raise RuntimeError("Unexpected artifact manifest")
for relative_name, expected in manifest["sha256"].items():
    path = (ARTIFACT_ROOT / relative_name).resolve(strict=True)
    if not path.is_relative_to(ARTIFACT_ROOT) or sha256_file(path) != expected:
        raise RuntimeError(f"Artifact hash mismatch: {relative_name}")

SOURCE_DIR = (ARTIFACT_ROOT / manifest["source_dir"]).resolve(strict=True)
PACKAGE_DIR = (ARTIFACT_ROOT / manifest["package_dir"]).resolve(strict=True)
if not SOURCE_DIR.is_relative_to(ARTIFACT_ROOT) or not PACKAGE_DIR.is_relative_to(ARTIFACT_ROOT):
    raise RuntimeError("Artifact source/package directory escapes the mounted Dataset")
metadata = json.loads((PACKAGE_DIR / "metadata.json").read_text(encoding="utf-8"))
if metadata.get("recipe") != RECIPE or metadata.get("mode") != "full":
    raise RuntimeError("Expected the trained full package")
if set(metadata.get("source_sha256", {})) != set(SOURCE_FILES):
    raise RuntimeError("Unexpected source manifest in model package")
for name in SOURCE_FILES:
    if sha256_file(SOURCE_DIR / name) != metadata["source_sha256"][name]:
        raise RuntimeError(f"Source differs from model training: {name}")
TEST_PATH = COMPETITION_DIR / "test.csv"
if not TEST_PATH.is_file():
    raise FileNotFoundError(f"Official competition test file missing: {TEST_PATH}")
print("Verified full model package and source hashes:", ARTIFACT_ROOT)

In [ ]:
import json
import zipfile

MODERN_RECIPE = 'e16_three_seed_modernbert_plus_e0c_roberta_row_z'
MODERN_MANIFEST_NAME = 'e16_artifact_manifest.json'
MODERN_ARTIFACT_ARCHIVE = None

if MODERN_ARTIFACT_ROOT is None:
    candidates = []
    for item in INPUT_ROOT.rglob(MODERN_MANIFEST_NAME):
        try:
            candidate = json.loads(item.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if candidate.get("recipe") == MODERN_RECIPE:
            candidates.append(item.parent)
    if not candidates:
        zip_candidates = []
        for item in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(item) as bundle:
                    if MODERN_MANIFEST_NAME in bundle.namelist():
                        zip_candidates.append(item)
            except (OSError, zipfile.BadZipFile):
                continue
        if MODERN_ARTIFACT_ZIP is not None:
            zip_candidates = [Path(MODERN_ARTIFACT_ZIP)]
        if len(zip_candidates) != 1:
            raise RuntimeError(
                f"Expected one E16 artifact ZIP, found {len(zip_candidates)}: {zip_candidates}. "
                "Set MODERN_ARTIFACT_ZIP in the config cell."
            )
        MODERN_ARTIFACT_ARCHIVE = Path(zip_candidates[0]).resolve(strict=True)
        extracted_root = Path("/kaggle/working/name_that_llm_e16_artifact")
        with zipfile.ZipFile(MODERN_ARTIFACT_ARCHIVE) as bundle:
            names = bundle.namelist()
            if len(names) != len(set(names)) or MODERN_MANIFEST_NAME not in names:
                raise RuntimeError("Invalid E16 artifact ZIP")
            if any(name.startswith("/") or ".." in name.split("/") or "\\" in name for name in names):
                raise RuntimeError("Unsafe path in E16 artifact ZIP")
            bundle.extractall(extracted_root)
        candidates = [extracted_root]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one E16 Dataset, found {len(candidates)}: {candidates}. "
            "Set MODERN_ARTIFACT_ROOT in the config cell."
        )
    MODERN_ARTIFACT_ROOT = candidates[0]

MODERN_ARTIFACT_ROOT = Path(MODERN_ARTIFACT_ROOT).resolve(strict=True)
modern_manifest = json.loads(
    (MODERN_ARTIFACT_ROOT / MODERN_MANIFEST_NAME).read_text(encoding="utf-8")
)
if modern_manifest.get("format_version") != 1 or modern_manifest.get("recipe") != MODERN_RECIPE:
    raise RuntimeError("Unexpected E16 artifact manifest")
if modern_manifest.get("source_file") != "source/fit_modernbert_full.py":
    raise RuntimeError("Unexpected E16 source path")
expected_package_dirs = {"42": "modern_seed42", "17": "modern_seed17", "73": "modern_seed73"}
if modern_manifest.get("package_dirs") != expected_package_dirs:
    raise RuntimeError("Unexpected E16 package directory mapping")
modern_hashes = modern_manifest.get("sha256")
if not isinstance(modern_hashes, dict) or not modern_hashes:
    raise RuntimeError("E16 artifact file hash manifest is missing")
actual_files = {item.relative_to(MODERN_ARTIFACT_ROOT).as_posix()
                for item in MODERN_ARTIFACT_ROOT.rglob("*") if item.is_file()}
if actual_files != set(modern_hashes) | {MODERN_MANIFEST_NAME}:
    raise RuntimeError("E16 artifact files differ from manifest")
for relative_name, expected_hash in modern_hashes.items():
    path = (MODERN_ARTIFACT_ROOT / relative_name).resolve(strict=True)
    if not path.is_relative_to(MODERN_ARTIFACT_ROOT) or sha256_file(path) != expected_hash:
        raise RuntimeError(f"E16 artifact hash mismatch: {relative_name}")

MODERN_SOURCE_PATH = (MODERN_ARTIFACT_ROOT / modern_manifest["source_file"]).resolve(strict=True)
MODERN_PACKAGE_DIRS = {
    int(seed): (MODERN_ARTIFACT_ROOT / directory).resolve(strict=True)
    for seed, directory in expected_package_dirs.items()
}
for seed, package in MODERN_PACKAGE_DIRS.items():
    if not package.is_relative_to(MODERN_ARTIFACT_ROOT):
        raise RuntimeError(f"E16 seed {seed} package escapes Dataset root")
    trained = json.loads((package / "metadata.json").read_text(encoding="utf-8"))
    if trained.get("seed") != seed:
        raise RuntimeError(f"E16 seed directory/metadata mismatch: {seed}")
    if trained.get("source_sha256", {}).get("fit_modernbert_full.py") != sha256_file(MODERN_SOURCE_PATH):
        raise RuntimeError(f"E16 seed {seed} source differs from fitted source")
    smoke = json.loads((package / "smoke_verification.json").read_text(encoding="utf-8"))
    if (smoke.get("seed") != seed or smoke.get("all_predictions_equal") is not True
            or smoke.get("package_metadata_sha256") != sha256_file(package / "metadata.json")
            or not isinstance(smoke.get("max_abs_logit_difference"), (int, float))
            or not 0 <= smoke["max_abs_logit_difference"] <= 2e-5):
        raise RuntimeError(f"E16 seed {seed} has no valid reload smoke verification")
print("Verified E16 ModernBERT package/source hashes:", MODERN_ARTIFACT_ROOT)


In [ ]:
import json
import zipfile

E24_RECIPE = 'e24_audited_prvsiyan_v9_full_e0c_replacement'
E24_MANIFEST_NAME = 'e24_artifact_manifest.json'
if E24_ARTIFACT_ROOT is None:
    candidates = []
    for item in INPUT_ROOT.rglob(E24_MANIFEST_NAME):
        try:
            candidate = json.loads(item.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if candidate.get("recipe") == E24_RECIPE:
            candidates.append(item.parent)
    if not candidates:
        zip_candidates = []
        for item in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(item) as bundle:
                    if E24_MANIFEST_NAME in bundle.namelist():
                        zip_candidates.append(item)
            except (OSError, zipfile.BadZipFile):
                continue
        if E24_ARTIFACT_ZIP is not None:
            zip_candidates = [Path(E24_ARTIFACT_ZIP)]
        if len(zip_candidates) != 1:
            raise RuntimeError(f"Expected one E24 artifact ZIP, found {len(zip_candidates)}: {zip_candidates}")
        extracted = Path("/kaggle/working/name_that_llm_e24_artifact")
        with zipfile.ZipFile(zip_candidates[0]) as bundle:
            names = bundle.namelist()
            if (len(names) != len(set(names)) or E24_MANIFEST_NAME not in names
                    or any(name.startswith("/") or ".." in name.split("/") or "\\" in name
                           for name in names)):
                raise RuntimeError("Invalid E24 artifact ZIP")
            bundle.extractall(extracted)
        candidates = [extracted]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one E24 Dataset, found {len(candidates)}: {candidates}")
    E24_ARTIFACT_ROOT = candidates[0]

E24_ARTIFACT_ROOT = Path(E24_ARTIFACT_ROOT).resolve(strict=True)
e24_manifest = json.loads((E24_ARTIFACT_ROOT / E24_MANIFEST_NAME).read_text(encoding="utf-8"))
if (e24_manifest.get("format_version") != 1 or e24_manifest.get("recipe") != E24_RECIPE
        or e24_manifest.get("package_dir") != "e24_e0c"
        or e24_manifest.get("source_dir") != "source"):
    raise RuntimeError("Unexpected E24 artifact manifest")
e24_hashes = e24_manifest.get("sha256")
if not isinstance(e24_hashes, dict) or set(e24_hashes) != {
    "e24_e0c/metadata.json", "e24_e0c/e0c.joblib", "e24_e0c/smoke_reference.json",
    "source/fit_e24_full_e0c.py", "source/run_e24_prvsiyan_external_ablation.py",
}:
    raise RuntimeError("Unexpected E24 artifact file list")
actual_files = {p.relative_to(E24_ARTIFACT_ROOT).as_posix()
                for p in E24_ARTIFACT_ROOT.rglob("*") if p.is_file()}
if actual_files != set(e24_hashes) | {E24_MANIFEST_NAME}:
    raise RuntimeError("E24 artifact contents differ from manifest")
for name, digest in e24_hashes.items():
    path = (E24_ARTIFACT_ROOT / name).resolve(strict=True)
    if not path.is_relative_to(E24_ARTIFACT_ROOT) or sha256_file(path) != digest:
        raise RuntimeError(f"E24 artifact hash mismatch: {name}")
E24_PACKAGE_DIR = (E24_ARTIFACT_ROOT / "e24_e0c").resolve(strict=True)
E24_SOURCE_DIR = (E24_ARTIFACT_ROOT / "source").resolve(strict=True)
trained = json.loads((E24_PACKAGE_DIR / "metadata.json").read_text(encoding="utf-8"))
for name in ("fit_e24_full_e0c.py", "run_e24_prvsiyan_external_ablation.py"):
    if sha256_file(E24_SOURCE_DIR / name) != trained.get("source_sha256", {}).get(name):
        raise RuntimeError(f"E24 fitted source mismatch: {name}")
for name in ("run_e0.py", "run_style.py", "run_combined.py"):
    if sha256_file(SOURCE_DIR / name) != trained.get("source_sha256", {}).get(name):
        raise RuntimeError(f"E24 shared E0c source mismatch: {name}")
print("Verified small E24 E0c package/source hashes:", E24_ARTIFACT_ROOT)


In [ ]:
import importlib.metadata as importlib_metadata
import os
import platform
import shutil
import subprocess
import sys

WHEELHOUSE_ARCHIVE_SHA256 = "2c4dc134de5717ed8d974c4fb8405f44d848045c08b08706826436ec66480a84"

def installed_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def runtime_issues():
    issues = []
    exact = {
        "scikit-learn": metadata["sklearn_version"],
        "torch": metadata["torch_version"].split("+")[0],
        "transformers": metadata["transformers_version"],
    }
    for name, expected in exact.items():
        actual = installed_version(name)
        comparable = actual.split("+")[0] if actual and name == "torch" else actual
        if comparable != expected:
            issues.append(f"{name}: installed={actual}, required={expected}")
    for name in ("numpy", "pandas", "joblib", "safetensors", "pyarrow"):
        if installed_version(name) is None:
            issues.append(f"{name}: missing")
    return issues

PYTHON_EXECUTABLE = Path(sys.executable)
issues = runtime_issues()
if issues:
    if WHEELHOUSE_DIR is None:
        directory_candidates = [
            item.parent for item in INPUT_ROOT.rglob("requirements-kaggle-offline.txt")
            if (item.parent / "manifest.json").is_file()
        ]
        if len(directory_candidates) > 1:
            siblings = [item for item in directory_candidates if item.parent == ARTIFACT_ROOT.parent]
            if siblings:
                directory_candidates = siblings
        if len(directory_candidates) == 1:
            WHEELHOUSE_DIR = directory_candidates[0]
            print("Found extracted wheelhouse Dataset:", WHEELHOUSE_DIR)
        elif directory_candidates:
            raise RuntimeError(
                f"Found multiple wheelhouse directories: {directory_candidates}. "
                "Set WHEELHOUSE_DIR in the config cell."
            )
        else:
            if WHEELHOUSE_ZIP is None:
                wheelhouse_candidates = list(INPUT_ROOT.rglob("kaggle_wheelhouse_cp313_linux_x86_64.zip"))
                if ARTIFACT_ARCHIVE is not None:
                    siblings = [item for item in wheelhouse_candidates if item.parent == ARTIFACT_ARCHIVE.parent]
                    if siblings:
                        wheelhouse_candidates = siblings
                if len(wheelhouse_candidates) != 1:
                    raise RuntimeError(
                        f"Dependency mismatch: {issues}. Found {len(wheelhouse_candidates)} wheelhouse ZIPs: "
                        f"{wheelhouse_candidates}. Set WHEELHOUSE_DIR or WHEELHOUSE_ZIP."
                    )
                wheelhouse_zip = wheelhouse_candidates[0]
            else:
                wheelhouse_zip = Path(WHEELHOUSE_ZIP)
            wheelhouse_zip = wheelhouse_zip.resolve(strict=True)
            if sha256_file(wheelhouse_zip) != WHEELHOUSE_ARCHIVE_SHA256:
                raise RuntimeError("Wheelhouse ZIP SHA-256 differs from the validated upload")
            extracted_wheels = Path("/kaggle/working/name_that_llm_wheelhouse")
            with zipfile.ZipFile(wheelhouse_zip) as bundle:
                names = bundle.namelist()
                if len(names) != len(set(names)) or any(
                    "/" in name or "\\" in name or name in (".", "..") for name in names
                ):
                    raise RuntimeError("Unsafe or duplicate path in wheelhouse ZIP")
                bundle.extractall(extracted_wheels)
            WHEELHOUSE_DIR = extracted_wheels
            print("Extracted offline wheelhouse:", WHEELHOUSE_DIR)
    WHEELHOUSE_DIR = Path(WHEELHOUSE_DIR).resolve(strict=True)
    if not WHEELHOUSE_DIR.is_dir():
        raise NotADirectoryError(WHEELHOUSE_DIR)
    if sys.version_info[:2] != (3, 13) or sys.platform != "linux" or platform.machine() != "x86_64":
        raise RuntimeError(
            "Attached wheelhouse requires CPython 3.13 on Linux x86_64; "
            f"found Python {sys.version.split()[0]} on {sys.platform}/{platform.machine()}"
        )
    manifest_rows = json.loads((WHEELHOUSE_DIR / "manifest.json").read_text(encoding="utf-8"))
    expected_wheels = {row["file"]: row for row in manifest_rows}
    expected_names = set(expected_wheels) | {
        "manifest.json", "README.txt", "SHA256SUMS.txt", "requirements-kaggle-offline.txt"
    }
    if (
        len(manifest_rows) != 35
        or len(expected_wheels) != 35
        or any(not name.endswith(".whl") or "/" in name or "\\" in name for name in expected_wheels)
        or {item.name for item in WHEELHOUSE_DIR.iterdir()} != expected_names
    ):
        raise RuntimeError("Wheelhouse directory layout differs from its 35-wheel manifest")
    for name, item in expected_wheels.items():
        wheel = WHEELHOUSE_DIR / name
        if wheel.stat().st_size != item["bytes"] or sha256_file(wheel) != item["sha256"]:
            raise RuntimeError(f"Wheelhouse integrity check failed: {name}")
    print("Verified all 35 offline wheel hashes")
    requirement_file = WHEELHOUSE_DIR / "requirements-kaggle-offline.txt"
    if not requirement_file.is_file() or "torch==2.6.0+cpu" not in requirement_file.read_text(encoding="utf-8"):
        raise RuntimeError("Expected the validated CPU-only Kaggle wheelhouse requirements")
    # Kaggle's system Python may fail ensurepip. Build a pip-free venv, then
    # let a host pip >=22.3 manage that interpreter (pip's documented --python option).
    venv_dir = Path("/kaggle/working/name_that_llm_venv_nodeps")
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv_dir)], check=True)
    venv_python = venv_dir / "bin" / "python"
    if not venv_python.is_file():
        raise RuntimeError(f"Virtual environment interpreter missing: {venv_python}")
    pip_frontend = [sys.executable, "-m", "pip"]
    pip_probe = subprocess.run(pip_frontend + ["--version"], capture_output=True, text=True)
    if pip_probe.returncode:
        pip_command = shutil.which("pip")
        if pip_command is None:
            raise RuntimeError("No system pip is available to bootstrap the pip-free venv offline")
        pip_frontend = [pip_command]
    # Install every verified wheel explicitly. The hub package declares hf-xet
    # for remote transfers, but that wheel is absent and inference is local-only.
    wheel_paths = [str(WHEELHOUSE_DIR / name) for name in sorted(expected_wheels)]
    subprocess.run(
        pip_frontend + ["--python", str(venv_dir), "install", "--no-index", "--no-deps"]
        + wheel_paths,
        check=True,
    )
    probe = (
        "import sys, numpy, pandas, sklearn, joblib, torch, transformers, pyarrow, safetensors; "
        "from transformers import AutoConfig, AutoTokenizer; "
        f"assert sklearn.__version__ == {metadata['sklearn_version']!r}; "
        f"assert torch.__version__.split('+')[0] == {metadata['torch_version'].split('+')[0]!r}; "
        f"assert transformers.__version__ == {metadata['transformers_version']!r}; "
        "AutoConfig.from_pretrained(sys.argv[1], local_files_only=True); "
        "AutoTokenizer.from_pretrained(sys.argv[2], local_files_only=True, use_fast=True); "
        "print('Offline venv and local model loading ready:', torch.__version__)"
    )
    probe_environment = os.environ.copy()
    probe_environment["HF_HUB_OFFLINE"] = "1"
    probe_environment["TRANSFORMERS_OFFLINE"] = "1"
    probe_environment["HF_HUB_DISABLE_XET"] = "1"
    subprocess.run(
        [str(venv_python), "-c", probe, str(PACKAGE_DIR / "encoder_config"),
         str(PACKAGE_DIR / "tokenizer")],
        env=probe_environment, check=True,
    )
    PYTHON_EXECUTABLE = venv_python
else:
    print("Notebook runtime already matches the fitted package; using", PYTHON_EXECUTABLE)


In [ ]:
# Probe the ModernBERT config and tokenizer with the interpreter selected above.
probe_environment = os.environ.copy()
probe_environment["HF_HUB_OFFLINE"] = "1"
probe_environment["TRANSFORMERS_OFFLINE"] = "1"
probe_environment["HF_HUB_DISABLE_XET"] = "1"
probe_environment["TORCHDYNAMO_DISABLE"] = "1"
probe = (
    "import sys, transformers; from transformers import AutoConfig, AutoTokenizer; "
    "c=AutoConfig.from_pretrained(sys.argv[1], local_files_only=True); "
    "assert c.model_type == 'modernbert' and c.hidden_size == 768; "
    "AutoTokenizer.from_pretrained(sys.argv[2], local_files_only=True, use_fast=True); "
    "print('Local ModernBERT config and tokenizer ready:', transformers.__version__)"
)
subprocess.run(
    [str(PYTHON_EXECUTABLE), "-c", probe,
     str(MODERN_PACKAGE_DIRS[42] / "encoder_config"),
     str(MODERN_PACKAGE_DIRS[42] / "tokenizer")],
    env=probe_environment, check=True,
)


In [ ]:
import hashlib
import os
import shutil
import subprocess

WORK_SOURCE = Path("/kaggle/working/name_that_llm_source_e24")
shutil.copytree(SOURCE_DIR, WORK_SOURCE, dirs_exist_ok=True)
shutil.copy2(MODERN_SOURCE_PATH, WORK_SOURCE / "fit_modernbert_full.py")
for name in ("fit_e24_full_e0c.py", "run_e24_prvsiyan_external_ablation.py"):
    shutil.copy2(E24_SOURCE_DIR / name, WORK_SOURCE / name)
WORK_SCRIPT = WORK_SOURCE / "kaggle" / "submit_e24_external_e0c.py"
WORK_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
WORK_SCRIPT.write_text('#!/usr/bin/env python3\n"""Offline E24 submission: audited external E0c plus unchanged E16 neural streams.\n\nEvery component is standardized across its 18 class scores within each row.\nThe ModernBERT stream is the mean of three separately standardized seeds;\nthe final score is the equal-weight sum of the three model-family streams.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport sys\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom final_pipeline import load_test, predict_package  # noqa: E402\nfrom fit_modernbert_full import (  # noqa: E402\n    LABELS as MODERN_LABELS,\n    SMOKE_TEXTS,\n    load_package as load_modern_package,\n    score_texts as score_modern_texts,\n)\nfrom run_e0 import LABELS  # noqa: E402\n\n\nEXPECTED_SEEDS = (42, 17, 73)\nEXPECTED_TRAIN_SHA256 = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nEXPECTED_MODEL_ID = "answerdotai/ModernBERT-base"\nEXPECTED_REVISION = "8949b909ec900327062f0ebf497f51aef5e6f0c8"\nE24_RECIPE = "e24_audited_prvsiyan_v9_full_e0c_replacement"\nE24_SOURCE_FILES = (\n    "fit_e24_full_e0c.py", "run_e24_prvsiyan_external_ablation.py",\n    "run_e0.py", "run_style.py", "run_combined.py",\n)\n# Package creation already requires <=2e-5 on the training machine. The\n# submission may run on a different CPU build; keep a separate cross-runtime\n# smoke tolerance while still requiring identical argmax and all file hashes.\nCROSS_RUNTIME_SMOKE_ATOL = 1e-3\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load_e24_surface(path: Path, source_dir: Path):\n    """Check exact fitted source/data provenance and three-text reload smoke."""\n    path = path.resolve(strict=True)\n    metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))\n    required = {\n        "package_schema_version": 1,\n        "recipe": E24_RECIPE,\n        "train_sha256": EXPECTED_TRAIN_SHA256,\n        "n_official_rows": 2699,\n        "n_official_prompts": 150,\n        "n_external_rows": 3620,\n        "n_external_prompts": 250,\n        "labels": list(LABELS),\n        "external_to_official_weight_ratio": 0.25,\n        "selected_manifest_sha256": "d7f8b7c9b0480ec896777e667589bf7d2fe12e503aa8f867c0708d0656ad6191",\n    }\n    for key, value in required.items():\n        if metadata.get(key) != value:\n            raise ValueError(f"E24 E0c has unexpected {key}")\n    if metadata.get("sklearn_version") != sklearn.__version__:\n        raise ValueError("E24 E0c requires its fitted scikit-learn version")\n    counts = metadata.get("official_rows_by_class", {})\n    external_counts = metadata.get("external_rows_by_class", {})\n    totals = metadata.get("external_total_weight_by_class", {})\n    if (sum(counts.values()) != 2699 or counts.get("claude-fable-5") != 149\n            or any(counts.get(label) != 150 for label in LABELS if label != "claude-fable-5")\n            or sum(external_counts.values()) != 3620\n            or external_counts.get("qwen3.8-max") != 0):\n        raise ValueError("E24 E0c class counts differ from locked full fit")\n    for label in LABELS:\n        expected = 0.0 if label == "qwen3.8-max" else 0.25 * counts[label]\n        if not np.isclose(totals.get(label, np.nan), expected, atol=1e-10):\n            raise ValueError(f"E24 E0c external weight ratio failed for {label}")\n    sources = metadata.get("source_sha256", {})\n    if set(sources) != set(E24_SOURCE_FILES):\n        raise ValueError("E24 fitted source list changed")\n    for name in E24_SOURCE_FILES:\n        if sha256(source_dir / name) != sources[name]:\n            raise ValueError(f"E24 fitted source hash changed: {name}")\n    hashes = metadata.get("artifact_sha256", {})\n    if set(hashes) != {"e0c.joblib", "smoke_reference.json"}:\n        raise ValueError("E24 E0c artifact hash list changed")\n    for name, digest in hashes.items():\n        if sha256(path / name) != digest:\n            raise ValueError(f"E24 E0c file hash changed: {name}")\n    model = joblib.load(path / "e0c.joblib")\n    classes = model.named_steps["classifier"].classes_\n    if set(classes) != set(LABELS):\n        raise ValueError("E24 E0c lost an official class")\n    positions = [int(np.flatnonzero(classes == label)[0]) for label in LABELS]\n    reference = json.loads((path / "smoke_reference.json").read_text(encoding="utf-8"))\n    if reference.get("labels") != list(LABELS) or len(reference.get("texts", [])) != 3:\n        raise ValueError("E24 E0c smoke reference changed")\n    expected = np.asarray(reference.get("logits"), dtype=np.float64)\n    actual = np.asarray(model.decision_function(reference["texts"])[:, positions], dtype=np.float64)\n    if expected.shape != (3, len(LABELS)) or not np.isfinite(actual).all():\n        raise ValueError("E24 E0c smoke logits invalid")\n    if not np.allclose(actual, expected, rtol=0, atol=1e-5):\n        raise ValueError("E24 E0c smoke logits changed after reload")\n    if not np.array_equal(actual.argmax(axis=1), expected.argmax(axis=1)):\n        raise ValueError("E24 E0c smoke predictions changed after reload")\n    return model, positions\n\n\ndef row_standardize(scores: np.ndarray, n_rows: int) -> np.ndarray:\n    array = np.asarray(scores, dtype=np.float64)\n    if array.shape != (n_rows, len(LABELS)) or not np.isfinite(array).all():\n        raise ValueError(f"Invalid or non-finite 18-class scores: {array.shape}")\n    centered = array - array.mean(axis=1, keepdims=True)\n    scale = centered.std(axis=1, keepdims=True)\n    if (scale <= 0).any() or not np.isfinite(scale).all():\n        raise ValueError("Constant or non-finite 18-class score row")\n    return centered / scale\n\n\ndef validate_modern_metadata(path: Path, metadata: dict, expected_seed: int) -> None:\n    if not isinstance(metadata, dict):\n        raise ValueError(f"ModernBERT metadata is not an object: {path}")\n    required = {\n        "package_schema_version": 1,\n        "mode": "full",\n        "seed": expected_seed,\n        "model_id": EXPECTED_MODEL_ID,\n        "requested_revision": EXPECTED_REVISION,\n        "pretrained_revision": EXPECTED_REVISION,\n        "train_sha256": EXPECTED_TRAIN_SHA256,\n        "n_training_rows": 2699,\n        "n_training_prompts": 150,\n        "labels": list(LABELS),\n    }\n    for key, expected in required.items():\n        if metadata.get(key) != expected:\n            raise ValueError(f"ModernBERT seed {expected_seed} has unexpected {key}: {path}")\n    if not metadata.get("artifact_sha256") or not metadata.get("source_sha256"):\n        raise ValueError(f"ModernBERT package hashes missing: {path}")\n    config = metadata.get("training_config", {})\n    for key, expected in (\n        ("epochs", 12), ("max_length", 256), ("batch_size", 16),\n        ("dropout", 0.1), ("pooling", "mean"),\n    ):\n        if config.get(key) != expected:\n            raise ValueError(f"ModernBERT seed {expected_seed} has unexpected {key}: {path}")\n    runtime = metadata.get("runtime_versions", {})\n    if runtime.get("torch", "").split("+")[0] != "2.6.0" or runtime.get("transformers") != "4.51.3":\n        raise ValueError(f"ModernBERT seed {expected_seed} runtime differs from fitted runtime")\n\n\ndef submit(\n    full_package: Path,\n    modern_packages: tuple[Path, Path, Path],\n    e24_package: Path,\n    test_path: Path,\n    output_path: Path,\n    batch_size: int = 16,\n) -> None:\n    if output_path.exists():\n        raise ValueError(f"Submission already exists: {output_path}")\n    if batch_size < 1:\n        raise ValueError("Batch size must be positive")\n    if len(MODERN_LABELS) != len(LABELS) or list(MODERN_LABELS) != list(LABELS):\n        raise ValueError("ModernBERT label order differs from the official fixed label order")\n    paths = tuple(path.resolve(strict=True) for path in modern_packages)\n    if len(set(paths)) != len(paths):\n        raise ValueError("Each ModernBERT seed must use a different package directory")\n    for expected_seed, path in zip(EXPECTED_SEEDS, paths, strict=True):\n        with (path / "metadata.json").open("r", encoding="utf-8") as handle:\n            declared_metadata = json.load(handle)\n        validate_modern_metadata(path, declared_metadata, expected_seed)\n    old_metadata = json.loads((full_package / "metadata.json").read_text(encoding="utf-8"))\n    for key, expected in (\n        ("train_sha256", EXPECTED_TRAIN_SHA256),\n        ("n_training_rows", 2699),\n        ("n_training_prompts", 150),\n    ):\n        if old_metadata.get(key) != expected:\n            raise ValueError(f"E0c/RoBERTa package has unexpected {key}")\n\n    test = load_test(test_path)\n    texts = test["text"].tolist()\n    n_rows = len(test)\n    print(f"Scoring {n_rows} official test rows", flush=True)\n\n    e24_surface, e24_positions = load_e24_surface(e24_package, Path(__file__).resolve().parents[1])\n    e24_logits = np.asarray(e24_surface.decision_function(texts)[:, e24_positions], dtype=np.float64)\n    _, old_surface_logits, roberta_logits = predict_package(full_package, texts, "full")\n    if old_surface_logits.shape != e24_logits.shape:\n        raise ValueError("E24 E0c and original E0c class score shapes differ")\n    family_score = row_standardize(e24_logits, n_rows)\n    family_score += row_standardize(roberta_logits, n_rows)\n    del e24_surface, e24_logits, old_surface_logits, roberta_logits\n    gc.collect()\n    print("E24 E0c and unchanged RoBERTa streams complete", flush=True)\n\n    modern_stream = np.zeros_like(family_score)\n    for expected_seed, path in zip(EXPECTED_SEEDS, paths, strict=True):\n        model, tokenizer, checked_metadata = load_modern_package(path, device="cpu")\n        validate_modern_metadata(path, checked_metadata, expected_seed)\n        reference = json.loads((path / "smoke_reference.json").read_text(encoding="utf-8"))\n        if reference.get("texts") != list(SMOKE_TEXTS):\n            raise ValueError(f"ModernBERT seed {expected_seed} smoke texts changed")\n        expected_smoke = np.asarray(reference.get("logits"), dtype=np.float32)\n        actual_smoke = score_modern_texts(\n            model, tokenizer, list(SMOKE_TEXTS), batch_size=2, device="cpu"\n        )\n        if expected_smoke.shape != actual_smoke.shape or not np.allclose(\n            expected_smoke, actual_smoke, rtol=0, atol=CROSS_RUNTIME_SMOKE_ATOL\n        ):\n            raise RuntimeError(f"ModernBERT seed {expected_seed} failed reload smoke check")\n        if not np.array_equal(expected_smoke.argmax(axis=1), actual_smoke.argmax(axis=1)):\n            raise RuntimeError(f"ModernBERT seed {expected_seed} changed smoke predictions")\n        logits = score_modern_texts(\n            model, tokenizer, texts, batch_size=batch_size, device="cpu"\n        )\n        modern_stream += row_standardize(logits, n_rows) / len(EXPECTED_SEEDS)\n        del model, tokenizer, logits\n        gc.collect()\n        print(f"ModernBERT seed {expected_seed} complete", flush=True)\n\n    combined = family_score + modern_stream\n    predictions = np.asarray(LABELS, dtype=object)[combined.argmax(axis=1)]\n    output = pd.DataFrame({"id": test["id"], "model": predictions})\n    if list(output.columns) != ["id", "model"] or len(output) != n_rows:\n        raise RuntimeError("Submission schema or row count changed")\n    if not output["id"].equals(test["id"]) or not set(output["model"]).issubset(LABELS):\n        raise RuntimeError("Submission IDs or labels are invalid")\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_name(output_path.name + ".tmp")\n    if temporary.exists():\n        raise ValueError(f"Stale temporary submission exists: {temporary}")\n    try:\n        output.to_parquet(temporary, index=False, engine="pyarrow")\n        roundtrip = pd.read_parquet(temporary, engine="pyarrow")\n        if not roundtrip.equals(output):\n            raise RuntimeError("Parquet round trip changed IDs or predictions")\n        temporary.replace(output_path)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    print(f"E24 offline submission written: {output_path} ({n_rows} rows)", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--full-package", required=True, type=Path)\n    for seed in EXPECTED_SEEDS:\n        parser.add_argument(f"--modern-seed{seed}", required=True, type=Path)\n    parser.add_argument("--e24-package", required=True, type=Path)\n    parser.add_argument("--test", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--modern-batch-size", type=int, default=16)\n    args = parser.parse_args()\n    submit(\n        args.full_package,\n        (args.modern_seed42, args.modern_seed17, args.modern_seed73),\n        args.e24_package,\n        args.test,\n        args.output,\n        args.modern_batch_size,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
if hashlib.sha256(WORK_SCRIPT.read_bytes()).hexdigest() != "da06935ebfd5963c7e6c9a81dc5515bf60438d3cbe876db5325b02745912d8f2":
    raise RuntimeError("E24 submission script hash mismatch")

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
environment = os.environ.copy()
environment["HF_HUB_OFFLINE"] = "1"
environment["HF_HUB_DISABLE_XET"] = "1"
environment["TRANSFORMERS_OFFLINE"] = "1"
environment["TOKENIZERS_PARALLELISM"] = "false"
environment["TORCHDYNAMO_DISABLE"] = "1"
environment["CUDA_VISIBLE_DEVICES"] = ""
threads = str(min(4, os.cpu_count() or 4))
environment["OMP_NUM_THREADS"] = threads
environment["MKL_NUM_THREADS"] = threads
command = [
    str(PYTHON_EXECUTABLE), str(WORK_SCRIPT),
    "--full-package", str(PACKAGE_DIR),
    "--modern-seed42", str(MODERN_PACKAGE_DIRS[42]),
    "--modern-seed17", str(MODERN_PACKAGE_DIRS[17]),
    "--modern-seed73", str(MODERN_PACKAGE_DIRS[73]),
    "--e24-package", str(E24_PACKAGE_DIR),
    "--test", str(TEST_PATH),
    "--output", str(OUTPUT_PATH),
]
subprocess.run(command, cwd=WORK_SOURCE, env=environment, check=True, timeout=8 * 60 * 60)
if not OUTPUT_PATH.is_file():
    raise RuntimeError("Inference completed without submission.parquet")
subprocess.run(
    [str(PYTHON_EXECUTABLE), "-c",
     "import sys, pyarrow.parquet as pq; print('Ready:', sys.argv[1], 'rows:', pq.read_metadata(sys.argv[1]).num_rows)",
     str(OUTPUT_PATH)], check=True,
)
